# Proyecto Final — Módulo 2 (Bootcamp Data & IA)

> Flujo de datos **end-to-end**: de un CSV de Kaggle a una app de Streamlit, pasando por una base de datos en la nube.

---

## Contexto

Hasta ahora habéis tocado por separado: SQL, Python, conexión a bases de datos, ETL, EDA y Streamlit. En este proyecto **integráis todo en un único flujo**, como lo haría un data analyst junior en su primer encargo.

## Objetivo

Partir de un dataset público de Kaggle, modelarlo en una base de datos relacional en la nube (TiDB), conectaros desde Python, construir un dataset analítico, hacer un EDA y exponer los resultados en una app de Streamlit.

## Tiempo y formato

- **2 sesiones de 3h30** para desarrollarlo.
- **1 sesión adicional** para presentar (5-10 min por alumno).
- Trabajo individual.

## Entregables

1. **Scripts SQL** con el `CREATE TABLE` de vuestras tablas (con PK y FK).
2. **Notebook Python** con la carga del CSV, extracción desde la BBDD, limpieza y EDA.
3. **App Streamlit** que muestre los hallazgos principales.
4. **Presentación** de 5-10 minutos ante el grupo.


## Esquema del flujo

```
  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
  │ CSV Kaggle   │───▶│  TiDB Cloud  │───▶│   Python     │
  │ (descarga)   │    │  (CREATE +   │    │ (mysql.      │
  │              │    │   INSERT)    │    │  connector)  │
  └──────────────┘    └──────────────┘    └──────┬───────┘
                                                  │
                                                  ▼
                                          ┌──────────────┐
                                          │  Dataset     │
                                          │  analítico   │
                                          │  (JOIN SQL)  │
                                          └──────┬───────┘
                                                  │
                                                  ▼
                                          ┌──────────────┐
                                          │  Limpieza    │
                                          │  + EDA       │
                                          └──────┬───────┘
                                                  │
                                                  ▼
                                          ┌──────────────┐
                                          │  Streamlit   │
                                          │  (dashboard) │
                                          └──────────────┘
```

**Idea clave:** cada fase produce un *output* que alimenta la siguiente. Si una fase se atasca, no podéis avanzar — por eso conviene **avanzar en horizontal** (algo mínimo en cada fase) antes que perfeccionar una sola.


---

## Fase 1 — Elegir dataset (15-30 min)

Tenéis **tres opciones recomendadas**. Todas vienen con varios CSV ya separados, lo que os ahorra decidir cómo dividir las tablas: la estructura ya está pensada.

### Opción A — Olist Brazilian E-Commerce (recomendado si os va lo de negocio)

- **Enlace:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- **Qué hay:** ventas reales de un marketplace brasileño (2016-2018).
- **Usar 3 CSV:** `olist_customers_dataset.csv`, `olist_orders_dataset.csv`, `olist_order_items_dataset.csv`.
- **Relaciones:** `customers` 1—N `orders` 1—N `order_items`.
- **Buenas preguntas:** ¿qué estados/ciudades generan más ventas? ¿cuánto tarda un pedido de media? ¿qué categorías facturan más?

### Opción B — Spotify Dataset 1921-2020 (recomendado si os va la música)

- **Enlace:** https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks
- **Qué hay:** ~600k tracks con features de audio (energy, danceability, valence...) y catálogo de artistas.
- **Usar 2 CSV:** `tracks.csv` y `artists.csv`.
- **Relaciones:** `artists` 1—N `tracks` (a través del campo artists/id_artists).
- **Buenas preguntas:** ¿cómo ha evolucionado la energía/duración de las canciones por década? ¿qué géneros dominan en cada época? ¿hay correlación entre popularidad y features?
- **Nota:** 600k tracks es mucho — **muestread a 50-100k filas** antes de subir.

### Opción C — TMDB 5000 Movies (recomendado si os va el cine)

- **Enlace:** https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata
- **Qué hay:** 5000 películas con presupuesto, ingresos, géneros, ratings, reparto.
- **Usar 2 CSV:** `tmdb_5000_movies.csv` y `tmdb_5000_credits.csv`.
- **Relaciones:** `movies` 1—1 `credits` (mismo `movie_id`).
- **Buenas preguntas:** ¿qué géneros son más rentables? ¿ha cambiado el presupuesto medio por década? ¿hay directores que garanticen taquilla?
- **Ojo:** algunas columnas (`genres`, `cast`) vienen como JSON dentro de un campo — toca parsearlas en la fase de limpieza.

### ¿Y si quiero otro dataset?

Permitido, **siempre que cumpla**:
1. Tenga al menos **2 CSV con una FK clara** entre ellos.
2. El total no exceda **200k filas** (para que la carga no se eternice).
3. Lo aprobéis conmigo antes de empezar.


---

## Fase 2 — Crear cuenta en TiDB Cloud (10 min)

Usaremos TiDB Cloud Starter (antes "Serverless"), la BBDD MySQL-compatible **gratuita y sin tarjeta** que ya vimos en el proyecto guiado de SQL.

### Pasos

1. Ir a https://tidbcloud.com y crear cuenta (con Google/GitHub es lo más rápido).
2. Crear un cluster tipo **Starter** (gratis, ~10 segundos).
3. En el cluster, ir a **Connect** → seleccionar **General** y **Python (mysql-connector)** → copiar los datos de conexión:
   - `host`
   - `port` (siempre 4000)
   - `user`
   - `password` (¡guardadla en cuanto se genere, no se vuelve a mostrar!)
   - `database` (creáis vosotros una; recomiendo el nombre `proyecto_final`)
4. Crear la base de datos desde el **SQL Editor** del cluster:

```sql
CREATE DATABASE proyecto_final;
```

### Límites del free tier (para que tengáis idea)

- 5 GiB de storage por cluster
- 50M Request Units / mes
- Hasta 5 clusters por organización

Con cualquiera de los 3 datasets recomendados vais sobrados.

### Guardar credenciales (sin subirlas a git)

Crear un archivo `credenciales.py` (en la misma carpeta del notebook):

```python
# credenciales.py — NO subir nunca a git/Drive público
mysql_config = {
    "host": "gateway01.eu-central-1.prod.aws.tidbcloud.com",
    "port": 4000,
    "user": "XXXXX.root",
    "password": "XXXXX",
    "database": "proyecto_final",
    "ssl_disabled": False,
}
```

Y luego, en cualquier notebook: `from credenciales import mysql_config`.


---

## Fase 3 — Modelar y crear las tablas en SQL (30-45 min)

### Qué tenéis que hacer

1. Abrir los CSV elegidos en pandas y mirar columnas, tipos y valores nulos.
2. Decidir **qué columnas conservar** (no hace falta meter todas — quedaos con las que aportan al análisis).
3. Escribir el `CREATE TABLE` para cada CSV, con:
   - Una **PRIMARY KEY** clara.
   - Una **FOREIGN KEY** entre tablas relacionadas.
   - Tipos de dato apropiados (`VARCHAR(n)`, `INT`, `DECIMAL(p,s)`, `DATE`, `DATETIME`...).
4. Ejecutar el SQL en el **SQL Editor** de TiDB.

### Plantilla de CREATE TABLE

```sql
-- Tabla padre (no tiene FK)
CREATE TABLE customers (
    customer_id   VARCHAR(50) PRIMARY KEY,
    customer_city VARCHAR(100),
    customer_state VARCHAR(5)
);

-- Tabla hija (con FK)
CREATE TABLE orders (
    order_id      VARCHAR(50) PRIMARY KEY,
    customer_id   VARCHAR(50),
    order_status  VARCHAR(20),
    order_date    DATETIME,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
```

### Trucos para no atascarse

- **Mirad primero los CSV con pandas** (`df.dtypes`, `df.head()`, `df.isna().sum()`) **antes** de escribir el SQL — así sabéis qué tipos usar.
- **Crear primero la tabla padre, luego las hijas** (si no, la FK falla).
- `VARCHAR` con un poco de margen: si el id más largo tiene 32 caracteres, poned `VARCHAR(50)`.
- Si un CSV trae IDs como números enteros enormes, usad `BIGINT` en vez de `INT`.
- Guardad el SQL en un archivo `schema.sql` aparte (es entregable).


---

## Fase 4 — Cargar los CSV en TiDB desde Python (30-45 min)

Aquí reutilizáis el patrón del proyecto guiado de SQL: `mysql.connector` + cursor. Lo único nuevo es `executemany`, que carga muchas filas de golpe en vez de una a una.

### Helper de carga

Pegad la función `cargar_csv_en_tabla` de abajo en una celda. Os sirve para cualquier dataset.

In [ ]:
import pandas as pd
import mysql.connector
from credenciales import mysql_config


def cargar_csv_en_tabla(df, tabla, conn, batch=1000):
    """Inserta un DataFrame en una tabla de TiDB en lotes de `batch` filas."""
    cursor = conn.cursor()
    cols = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO {tabla} ({cols}) VALUES ({placeholders})"

    filas = [tuple(None if pd.isna(v) else v for v in row)
             for row in df.itertuples(index=False, name=None)]

    for i in range(0, len(filas), batch):
        cursor.executemany(sql, filas[i:i+batch])
        conn.commit()
        print(f"  insertadas {min(i+batch, len(filas))}/{len(filas)}")
    cursor.close()


# Ejemplo de uso (adaptar columnas a vuestro CSV):
# df_customers = pd.read_csv("olist_customers_dataset.csv")
# df_customers = df_customers[["customer_id", "customer_city", "customer_state"]]
# conn = mysql.connector.connect(**mysql_config)
# cargar_csv_en_tabla(df_customers, "customers", conn)
# conn.close()

### Trucos

- **Cargad primero la tabla padre, luego las hijas** (igual que al crear). Si insertáis una orden con un `customer_id` que no existe en `customers`, la FK os echará.
- **Limpiad antes de insertar**: convertir fechas con `pd.to_datetime`, quitar columnas innecesarias, descartar filas con FK nula.
- Si el dataset es grande (>100k filas), **muestread** con `df.sample(n=50000, random_state=42)`.
- Si os da error de tipo, mirad cómo es la columna en pandas (`df.dtypes`) y en SQL (`DESCRIBE tabla;`). El 90% de errores son fechas mal parseadas o textos demasiado largos.

### Plan B — si os atascáis con la carga

Si el helper de Python os da problemas y se os va el tiempo, podéis cargar el CSV **directamente desde la UI de TiDB**: en vuestro cluster, pestaña **Import** → **Local File** → arrastrar el CSV (hasta 250 MiB). TiDB detecta tipos y crea/puebla la tabla por vosotros.

**Condición:** si recurrís a esta vía, tenéis que **mencionarlo explícitamente en la presentación** — qué error os bloqueó y cómo lo habríais resuelto con más tiempo. No pasa nada por usar el plan B; sí pasa por ocultarlo.


---

## Fase 5 — Extraer el dataset analítico (15-30 min)

Aquí volvéis a usar SQL **de verdad**: una sola query con `JOIN` que os devuelva la tabla plana que vais a analizar. **No** hagáis `SELECT * FROM tabla` y juntéis en pandas — eso desaprovecha la BBDD.

### Patrón

In [ ]:
conn = mysql.connector.connect(**mysql_config)

query = """
SELECT
    o.order_id,
    o.order_date,
    o.order_status,
    c.customer_city,
    c.customer_state,
    oi.price,
    oi.freight_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, conn)
conn.close()

df.head()

### Buenas prácticas

- **Una sola query** que devuelva todo lo necesario para el EDA.
- Filtrar en SQL (`WHERE`) lo que claramente sobra (estados cancelados, fechas fuera de rango, etc.).
- Guardar el `df` en un `.parquet` o `.pickle` local para no tener que reconectaros cada vez:

```python
df.to_parquet("dataset_analitico.parquet")
# Luego: df = pd.read_parquet("dataset_analitico.parquet")
```


---

## Fase 6 — Limpieza y preprocesamiento (30-45 min)

### Checklist mínimo

- [ ] **Tipos correctos**: fechas como `datetime`, números como `float`/`int`, categóricas como `category`.
- [ ] **Nulos**: identificar con `df.isna().sum()`. Decidir uno a uno: ¿imputo, descarto fila o descarto columna? Justificadlo.
- [ ] **Duplicados**: `df.duplicated().sum()` → `df.drop_duplicates()` si toca.
- [ ] **Outliers obvios**: precios negativos, fechas en el año 1900, edades de 200 años...
- [ ] **Columnas derivadas**: año, mes, día de la semana, categoría agrupada, etc. — lo que necesitéis para el EDA.

### Tip de oro

**No limpiéis todo el dataset "por si acaso"**. Limpiad sólo lo que afecta a las preguntas que vais a responder. Si una columna no la vais a usar, no hace falta tratar sus nulos.


---

## Fase 7 — EDA (45-60 min)

El EDA es la fase donde **descubrís la historia** del dataset. No es "hacer gráficos sueltos", es **responder preguntas**.

### Estructura recomendada

1. **Descripción general** (5 min): cuántas filas, qué periodo, qué variables.
2. **Univariante** (15 min): distribución de cada variable importante (histogramas, boxplots, value_counts).
3. **Bivariante** (20 min): relación entre variables (scatter, barras por grupo, correlaciones).
4. **Hallazgos** (10 min): 3-5 conclusiones claras con un gráfico cada una.

### Preguntas guía para arrancar (escoged 3-5 según vuestro dataset)

- ¿Cómo evoluciona X a lo largo del tiempo?
- ¿Qué categoría/grupo domina? ¿Hay desigualdad fuerte?
- ¿Hay correlación entre dos variables clave?
- ¿Existen segmentos diferenciados (clusters obvios)?
- ¿Qué porcentaje del total acumulan el top-10 / top-20%? (Pareto)

### Tipos de gráfico por situación

| Lo que quieres mostrar | Gráfico |
|---|---|
| Distribución de una variable numérica | histograma, boxplot |
| Frecuencia de una categórica | barras horizontales (`value_counts().plot.barh()`) |
| Evolución temporal | línea (mejor con `resample('M')` o `groupby(año)`) |
| Relación numérica-numérica | scatter (con `alpha=0.3` si hay muchos puntos) |
| Comparar grupos | boxplot por categoría, barras agrupadas |
| Matriz de correlaciones | heatmap (`sns.heatmap(df.corr())`) |
| Datos geográficos | mapa con `plotly.express.scatter_mapbox` o `folium` |

### Cómo conseguir un buen EDA (no sólo "correcto")

- **Cada gráfico tiene que responder una pregunta concreta**. Si no sabéis qué pregunta responde, sobra.
- **Título descriptivo en cada gráfico**: "Distribución de precios" está mal. "El 80% de los pedidos cuesta menos de 100 €" está bien.
- **Insight escrito al lado de cada gráfico**: una frase explicando qué se ve y qué implica.
- **Storytelling**: que el notebook se lea como una historia (cada apartado lleva al siguiente).


---

## Fase 8 — Dashboard con Streamlit (60-90 min)

El Streamlit **no repite el EDA**: presenta los hallazgos a un público no técnico, de forma interactiva.

### Estructura mínima recomendada

1. **Título + intro** (1 párrafo): qué dataset, qué se analiza.
2. **Métricas clave** (`st.metric`): 3-4 números grandes (total ventas, nº clientes, etc.).
3. **Filtros** en la sidebar (`st.sidebar.selectbox`, `st.sidebar.slider`): que el usuario pueda explorar.
4. **2-4 gráficos** que reflejen los hallazgos del EDA, **filtrados** según la sidebar.
5. **Conclusiones** al final.

### Boilerplate

In [ ]:
# Guardar como app.py y lanzar con:  streamlit run app.py

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Mi Proyecto", layout="wide")

@st.cache_data
def cargar_datos():
    return pd.read_parquet("dataset_analitico.parquet")

df = cargar_datos()

st.title("Análisis de ventas — Olist")
st.markdown("Exploración de pedidos del marketplace brasileño 2016-2018.")

# Sidebar con filtros
estados = st.sidebar.multiselect("Estado", df["customer_state"].unique(), default=["SP", "RJ"])
df_f = df[df["customer_state"].isin(estados)]

# Métricas
col1, col2, col3 = st.columns(3)
col1.metric("Pedidos", f"{len(df_f):,}")
col2.metric("Facturación", f"{df_f['price'].sum():,.0f} €")
col3.metric("Ticket medio", f"{df_f['price'].mean():.2f} €")

# Gráfico
ventas_estado = df_f.groupby("customer_state")["price"].sum().sort_values(ascending=False)
fig = px.bar(ventas_estado, title="Facturación por estado")
st.plotly_chart(fig, use_container_width=True)

### Cómo conseguir un Streamlit que destaque

- **Layout limpio**: usad `st.columns` para alinear las métricas, no las pongáis una debajo de otra.
- **`@st.cache_data` en la función que carga datos** — sin esto la app va lentísima.
- **Una métrica clara arriba** (la cifra más impactante del proyecto).
- **Filtros que tengan sentido**: si el filtro no cambia nada útil del gráfico, no lo pongáis.
- **No metáis 10 gráficos**. 3-4 buenos > 10 mediocres.
- **Pensad en el usuario final**: si fuera alguien de negocio que no sabe Python, ¿entendería lo que ve?


---

## Fase 9 — Presentación (5-10 min)

### Guion sugerido

1. **Contexto** (30s): qué dataset y por qué.
2. **Flujo técnico** (2 min): enseñad la BBDD en TiDB, la query con JOIN, el notebook.
3. **Hallazgos** (3-5 min): los 3 insights principales, con un gráfico cada uno.
4. **Demo Streamlit** (1-2 min): mostrad la app en directo, jugad con un filtro.
5. **Conclusión + dificultades** (1 min): qué habéis aprendido, qué fue lo más difícil.

### Errores típicos a evitar en la exposición

- Empezar mostrando código durante 5 minutos sin contar qué hace el proyecto.
- Leer el notebook celda por celda.
- No haber probado el Streamlit antes (y que falle en vivo).
- Resumir todo lo que hicisteis en vez de centrar en lo que **descubristeis**.


---

## Errores comunes y cómo salir de ellos

**"Error 1452: Cannot add or update a child row: foreign key constraint fails"**
→ Estás insertando en la tabla hija un ID que no existe en la padre. Carga primero la padre, o filtra el df hijo para quedarte sólo con IDs válidos.

**"Error 1366: Incorrect datetime value"**
→ La columna en pandas es string y la tabla espera DATETIME. Convierte con `pd.to_datetime(df['col'])` antes de insertar.

**"Error 1406: Data too long for column"**
→ Algún valor es más largo que el `VARCHAR(n)` que definiste. Mira el `df['col'].str.len().max()` y amplía.

**El INSERT tarda muchísimo**
→ Estás insertando fila a fila con `execute` en bucle. Usa el helper `cargar_csv_en_tabla` con `executemany`. Si aún así tarda mucho, baja el dataset con `df.sample()`.

**Streamlit recarga muy lento**
→ Falta `@st.cache_data` en la función que lee el parquet. Sin caché, recarga todo en cada interacción.

**"He cargado las tablas pero no veo nada en TiDB"**
→ Falta `conn.commit()` después del `executemany`. El helper ya lo hace, pero si lo escribiste tú, revisa.

---

## Recursos

- **TiDB Cloud:** https://tidbcloud.com — registro gratuito.
- **Documentación mysql-connector:** https://dev.mysql.com/doc/connector-python/en/
- **Documentación Streamlit:** https://docs.streamlit.io/
- **Galería Streamlit (para inspiración):** https://streamlit.io/gallery
- **Plotly Express:** https://plotly.com/python/plotly-express/


